# Phase 1C - Table-only RAG (Colab runner)

Runner only: mount, git pull, install, call scripts. No logic lives here
(see CLAUDE.md P1/P2). Edit `src/` and `scripts/` locally, push, then re-run.

Table-only RAG over the Phase 1A/1B tables: serialize -> chunk -> retrieve -> grounded
answer. GT-filled and OCR-filled corpora stay strictly separate (P4); gold answers come
from GT, and the eval runs each corpus independently to measure the OCR-vs-GT QA gap.

Mostly CPU/API; only embedding (FAISS index build) uses the GPU. The cells below cover
data prep (gt_filled headers, corpora, QA set) and BM25 retrieval eval - all CPU, no key.

## Boot

In [ ]:
# 1. Mount Drive (chunks / QA sets / rag index persist here).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Check out the ACTIVE dev branch and confirm HEAD.
#    `git pull` alone only updates the CURRENT branch, so if the VM is on main (e.g. after
#    an interim merge) it runs stale code. Pin the branch explicitly and verify the hash.
BRANCH = 'feature/phase1c-retrieval'   # set to 'main' once Phase 1C is merged
!cd /content/FinDocStructRAG && git fetch origin --quiet && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
!cd /content/FinDocStructRAG && echo 'branch:' $(git rev-parse --abbrev-ref HEAD) ' HEAD:' $(git log --oneline -1)
%cd /content/FinDocStructRAG

In [ ]:
# 3. Make src/ importable and sanity-check the Phase 1C paths.
import sys, importlib
sys.path.insert(0, '/content/FinDocStructRAG')
from src import config
importlib.reload(config)  # refresh if the kernel imported an older config before the pull
print('IN_COLAB :', config.IN_COLAB)
print('CHUNKS   :', config.CHUNKS)
print('QA_DIR   :', config.QA_DIR)
print('QA seed  :', config.QA_MANUAL_SEED, '(exists:', config.QA_MANUAL_SEED.exists(), ')')

## Step 0 - (re)build gt_filled WITH headers (CPU)

Header detection (is_header) was added to `normalize_tatr_prediction`, but the gt_filled
tables on Drive were built before it. `--force` reprocesses all samples (ignoring the
resume manifest) so the regenerated tables carry column headers - which the linearized
serialization and the templated QA generator both need. Expect `processed=300 skipped=0`
(if you see `skipped=300`, --force is not in the checked-out code - fix the branch above).

In [ ]:
!python scripts/run_phase1b_gt_filled.py --limit 300 --seed 42 --run-id mvp_rand --force

In [ ]:
# Header sanity + FAIL-FAST. If it stops, the two prints say which cause:
#   (b) column_headers = 0  -> the GT annotation lacks header boxes (parse/class-name issue)
#   (a) column_headers > 0 but 0 header cells -> the marking overlap missed
import json, importlib
from src import config, fintabnet_loader as fl
importlib.reload(fl)

xmls = fl.find_xml_files(fl.structure_root(), limit=1, seed=42)
p = fl.parse_structure_xml(xmls[0])
print('sample class_counts   :', p['class_counts'])
print('sample column_headers :', len(p['column_headers']), 'boxes')

sample = sorted(config.TABLES_GT_FILLED.glob('*.json'))[:30]
with_headers = sum(1 for q in sample
                   if any(c.get('is_header') for c in json.loads(q.read_text())['cells']))
print(f'gt_filled (first {len(sample)}): {with_headers} have header cells')

assert with_headers > 0, (
    'No header cells marked - stopping so an empty QA set is not produced. See the two '
    'prints above for the cause (no header boxes vs marking missed).')

## Step 0b - backfill ocr_filled headers (CPU, no OCR re-run)

ocr_filled was built from tatr_predicted, whose grid carries no header marking, so its
cells are all `is_header=False`. gt_filled now has headers, so a GT-vs-OCR linearized
comparison would be unfair: gt_linearized pairs each value with its header, ocr_linearized
cannot. This reads the `column_headers` boxes from `tatr_raw/<id>.json` (same TATR
coordinate space) and marks `is_header` with the same IoMin rule - a fairness fix, no OCR
re-run. Expect most tables to report `>=1 hdr`.

In [ ]:
!python scripts/mark_ocr_filled_headers.py

## Step 1 - build retrieval corpora (CPU)

Serializes every gt_filled / ocr_filled table into one chunk per table and writes a JSONL
corpus per (text_source, serialization) under `rag_index/chunks/`. GT and OCR corpora are
kept separate (P4). With headers present, the linearized corpus now pairs each value with
its column header.

In [ ]:
!python scripts/build_table_chunks.py

## Step 2 - QA dataset (templated + manual seed) (CPU)

Generates ~30 templated lookup/numeric questions from GT cells (answer = GT cell) and
merges the hand-authored seed `qa/qa_manual_seed.jsonl` if present. With headers fixed,
`templated` should be ~30. Re-run after you author + push the manual set.

In [ ]:
!python scripts/build_qa_dataset.py --limit 30 --seed 42

## Step 3 - retrieval evaluation (CPU, no LLM)

BM25 over each corpus ({gt,ocr} x {markdown,linearized}) with the QA set; reports
hit@k / recall@k / MRR@k so markdown-vs-linearized and GT-vs-OCR are directly comparable.
No API key (P5); answer generation is a later step. Writes
`outputs/evaluation/rag/phase1c_retrieval.json`.

In [ ]:
!python scripts/evaluate_rag.py

## Step 4 - browse tables for manual QA authoring (CPU)

Prints a seeded sample of GT-filled tables as markdown (reads like a real table) with each
`sample_id` / `chunk_id`, so you can read off true answers and write the manual +
unanswerable questions. Change `--limit` / `--seed` to see different tables.

In [ ]:
!python scripts/preview_chunks.py --limit 15 --seed 7

## Author the manual set, then re-run Steps 2-3

The ~30 templated questions are automatic; the harder + unanswerable ones are hand-authored:

1. From the Step 4 output, pick ~10 tables for comparison/reasoning questions and a few for
   plausible-but-unanswerable ones.
2. **Locally** in VS Code, add one JSON object per line to `qa/qa_manual_seed.jsonl`
   (schema + examples in `qa/README.md`), using a real `sample_id` you just read.
3. `git push` -> re-run Boot, then **Step 2** (merge) and **Step 3** (re-score).

Dense retrieval (bge + FAISS) and RRF fusion, then LLM answer generation, are added as those
land.